In [1]:
from pathlib import Path
import os
import time
import json
import csv
import math
import hashlib
import platform
import shutil

import numpy as np

from pynq import Overlay, allocate


# ============================================================
# CAMINHOS
# ============================================================

BASE = Path(
    "/home/xilinx/jupyter_notebooks/resnet8_hls_ip11"
)

BIT = BASE / "resnet8_hls_ip11.bit"
HWH = BASE / "resnet8_hls_ip11.hwh"
DATA = BASE / "cifar10_test_uint8.npz"

OUT = BASE / "resultados_final_10k100_2026-08-28"
OUT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# VERIFICAÇÕES
# ============================================================

print("EUID:", os.geteuid())
print("CPUs ARM visíveis:", os.cpu_count())

assert os.geteuid() == 0, (
    "Execute no kernel privilegiado/root do Jupyter."
)

assert BIT.exists(), BIT
assert HWH.exists(), HWH
assert DATA.exists(), DATA


# ============================================================
# OVERLAY
# ============================================================

print("\nCarregando Overlay...")

overlay = Overlay(
    str(BIT),
    download=True
)

print("IPs:")
print(overlay.ip_dict.keys())


dma = overlay.axi_dma_0


# ============================================================
# DATASET
# ============================================================

data = np.load(DATA)

x_test = data["x"]
y_test = data["y"].reshape(-1)

assert x_test.shape == (10000, 32, 32, 3)
assert y_test.shape == (10000,)
assert x_test.dtype == np.uint8

print("\nDataset:", x_test.shape, x_test.dtype)
print("Labels :", y_test.shape, y_test.dtype)

print("\nDistribuição:")
print(
    np.bincount(
        y_test,
        minlength=10
    )
)

print("\nOUT:")
print(OUT)

EUID: 0
CPUs ARM visíveis: 4

Carregando Overlay...


IPs:
dict_keys(['axi_dma_0', 'zynq_ultra_ps_e_0'])

Dataset: (10000, 32, 32, 3) uint8
Labels : (10000,) uint8

Distribuição:
[1000 1000 1000 1000 1000 1000 1000 1000 1000 1000]

OUT:
/home/xilinx/jupyter_notebooks/resnet8_hls_ip11/resultados_final_10k100_2026-08-28


In [2]:
# ============================================================
# FIXED POINT
# ============================================================

FIXED_W = 22
FIXED_I = 12
FIXED_F = FIXED_W - FIXED_I

SCALE = 1 << FIXED_F

MASK22 = (
    (1 << FIXED_W)
    - 1
)

SIGN22 = (
    1 << (FIXED_W - 1)
)

RAW_MIN = -(
    1 << (FIXED_W - 1)
)

RAW_MAX = (
    (1 << (FIXED_W - 1))
    - 1
)


# ============================================================
# TAMANHOS
# ============================================================

N_IMAGES = 10000
N_CLASSES = 10

N_PIXELS = 32 * 32

INPUT_BYTES = 16384
OUTPUT_BYTES = 64


# ============================================================
# BUFFERS PYNQ
# ============================================================

input_buffer = allocate(
    shape=(1024, 4),
    dtype=np.uint32
)

output_buffer = allocate(
    shape=(16,),
    dtype=np.uint32
)


# ============================================================
# PREPROCESSAMENTO
# ============================================================

def pack_image_to(
    image_u8,
    destination
):

    x = (
        image_u8.astype(np.float32)
        / np.float32(255.0)
    )

    q = np.rint(
        x * np.float32(SCALE)
    ).astype(np.int64)

    np.clip(
        q,
        RAW_MIN,
        RAW_MAX,
        out=q
    )

    q = (
        q & MASK22
    ).astype(np.uint32)

    q = q.reshape(
        N_PIXELS,
        3
    )

    destination[:, 0] = q[:, 0]
    destination[:, 1] = q[:, 1]
    destination[:, 2] = q[:, 2]
    destination[:, 3] = 0


# ============================================================
# DMA
# ============================================================

def dma_inference_only():

    # recv primeiro por segurança
    dma.recvchannel.transfer(
        output_buffer,
        nbytes=OUTPUT_BYTES
    )

    dma.sendchannel.transfer(
        input_buffer,
        nbytes=INPUT_BYTES
    )

    dma.sendchannel.wait()
    dma.recvchannel.wait()


# ============================================================
# DECODE
# ============================================================

def decode_logits():

    raw = (
        np.asarray(
            output_buffer[:10],
            dtype=np.uint32
        )
        & MASK22
    ).astype(np.int64)

    negative = (
        raw & SIGN22
    ) != 0

    raw[negative] -= (
        1 << FIXED_W
    )

    return (
        raw.astype(np.float32)
        / np.float32(SCALE)
    )


def dma_ok():

    return (
        not dma.sendchannel.error
        and
        not dma.recvchannel.error
    )


print("Buffers:")
print(
    "input :",
    input_buffer.shape,
    input_buffer.nbytes,
    "bytes"
)

print(
    "output:",
    output_buffer.shape,
    output_buffer.nbytes,
    "bytes"
)

Buffers:
input : (1024, 4) 16384 bytes
output: (16,) 64 bytes


In [3]:
pack_image_to(
    x_test[0],
    input_buffer
)

dma_inference_only()

logits = decode_logits()

pred = int(
    np.argmax(logits)
)

print("Esperada:", int(y_test[0]))
print("Predita :", pred)
print("DMA OK  :", dma_ok())

print("\nLogits:")
print(logits)

print("\nPadding:")
print(output_buffer[10:16])


assert pred == int(y_test[0])
assert dma_ok()

print("\nSMOKE TEST: OK")

Esperada: 3
Predita : 3
DMA OK  : True

Logits:
[ -7.7910156  -8.96582   -13.777344    3.600586  -10.802734    1.8779297
  -1.3779297 -13.140625   -5.9716797 -12.662109 ]

Padding:
[0 0 0 0 0 0]

SMOKE TEST: OK


In [4]:
WARMUP = 100

print(
    f"Warm-up: {WARMUP} inferências"
)

for i in range(WARMUP):

    pack_image_to(
        x_test[i % N_IMAGES],
        input_buffer
    )

    dma_inference_only()

    _ = decode_logits()


assert dma_ok()

print("Warm-up concluído.")

Warm-up: 100 inferências
Warm-up concluído.


In [5]:
predictions_accuracy = np.empty(
    N_IMAGES,
    dtype=np.uint8
)

logits_accuracy = np.empty(
    (N_IMAGES, N_CLASSES),
    dtype=np.float32
)


print(
    "===== ACURÁCIA COMPLETA CIFAR-10 ====="
)

t_acc0 = time.perf_counter()


for i in range(N_IMAGES):

    pack_image_to(
        x_test[i],
        input_buffer
    )

    dma_inference_only()

    logits = decode_logits()

    logits_accuracy[i] = logits

    predictions_accuracy[i] = int(
        np.argmax(logits)
    )

    if (
        (i + 1) % 1000
        == 0
    ):

        partial_correct = int(
            np.sum(
                predictions_accuracy[:i+1]
                ==
                y_test[:i+1]
            )
        )

        print(
            f"{i+1:5d}/{N_IMAGES} | "
            f"accuracy parcial = "
            f"{partial_correct/(i+1)*100:.3f}%"
        )


accuracy_elapsed = (
    time.perf_counter()
    - t_acc0
)


assert dma_ok()


correct = int(
    np.sum(
        predictions_accuracy
        ==
        y_test
    )
)

accuracy = (
    correct
    / N_IMAGES
)

===== ACURÁCIA COMPLETA CIFAR-10 =====
 1000/10000 | accuracy parcial = 75.700%
 2000/10000 | accuracy parcial = 74.550%
 3000/10000 | accuracy parcial = 74.133%
 4000/10000 | accuracy parcial = 74.225%
 5000/10000 | accuracy parcial = 74.780%
 6000/10000 | accuracy parcial = 75.133%
 7000/10000 | accuracy parcial = 75.143%
 8000/10000 | accuracy parcial = 75.100%
 9000/10000 | accuracy parcial = 74.878%
10000/10000 | accuracy parcial = 74.920%


In [6]:
# ============================================================
# MATRIZ DE CONFUSÃO
# linha = classe verdadeira
# coluna = classe prevista
# ============================================================

confusion_matrix = np.zeros(
    (N_CLASSES, N_CLASSES),
    dtype=np.int64
)

np.add.at(
    confusion_matrix,
    (
        y_test.astype(np.int64),
        predictions_accuracy.astype(np.int64)
    ),
    1
)


# ============================================================
# IC95% WILSON
# ============================================================

z = 1.959963984540054

n = N_IMAGES
phat = accuracy

denom = (
    1.0
    + z*z/n
)

center = (
    phat
    + z*z/(2*n)
) / denom

half = (
    z
    / denom
    * math.sqrt(
        phat*(1-phat)/n
        + z*z/(4*n*n)
    )
)

wilson_low = center - half
wilson_high = center + half


print("\n===== RESULTADO DE ACURÁCIA =====")

print(
    f"Acertos  : {correct}/{N_IMAGES}"
)

print(
    f"Accuracy : "
    f"{accuracy*100:.4f}%"
)

print(
    "Wilson 95%:",
    f"[{wilson_low*100:.4f}%, "
    f"{wilson_high*100:.4f}%]"
)

print(
    f"Tempo funcional total: "
    f"{accuracy_elapsed:.2f} s"
)

print("\nMatriz de confusão:")
print(confusion_matrix)


===== RESULTADO DE ACURÁCIA =====
Acertos  : 7492/10000
Accuracy : 74.9200%
Wilson 95%: [74.0609%, 75.7599%]
Tempo funcional total: 16.05 s

Matriz de confusão:
[[878   6  15   9  13   6  13  16  30  14]
 [ 45 853   7   5   1   4  12   7  12  54]
 [ 86   1 599  69  67  50  80  39   3   6]
 [ 28   3  40 624  45 146  74  34   3   3]
 [ 19   1  73  76 640  28  73  83   7   0]
 [ 12   2  29 152  21 679  36  64   4   1]
 [ 13   0  48  54  15  22 841   4   1   2]
 [ 18   1  30  49  33  55   7 801   3   3]
 [139  18   7  18   8   5   4   5 779  17]
 [ 61  49   7  25   4   6   5  27  18 798]]


In [7]:
np.save(
    OUT / "predictions_accuracy_10000.npy",
    predictions_accuracy
)

np.save(
    OUT / "logits_accuracy_10000.npy",
    logits_accuracy
)

np.save(
    OUT / "confusion_matrix_10000.npy",
    confusion_matrix
)


np.savetxt(
    OUT / "confusion_matrix_10000.csv",
    confusion_matrix,
    fmt="%d",
    delimiter=","
)


accuracy_summary = {

    "dataset":
        "CIFAR-10 official test set",

    "unique_images":
        N_IMAGES,

    "classes":
        N_CLASSES,

    "correct":
        correct,

    "accuracy":
        accuracy,

    "accuracy_percent":
        accuracy * 100.0,

    "wilson95_low":
        wilson_low,

    "wilson95_high":
        wilson_high,

    "wilson95_low_percent":
        wilson_low * 100.0,

    "wilson95_high_percent":
        wilson_high * 100.0,

    "elapsed_s":
        accuracy_elapsed,

    "model_output":
        "10 logits, no softmax",

    "prediction":
        "argmax(logits)",

    "preprocessing":
        "uint8 -> float32 /255 -> ap_fixed<22,12> packing",

    "batch":
        1,

    "note":
        (
            "Functional validation only. "
            "Elapsed time is not used as "
            "a latency or throughput benchmark."
        )
}


(
    OUT / "accuracy_10000.json"
).write_text(
    json.dumps(
        accuracy_summary,
        indent=2
    )
)


print("Arquivos salvos:")

for name in [
    "predictions_accuracy_10000.npy",
    "logits_accuracy_10000.npy",
    "confusion_matrix_10000.npy",
    "confusion_matrix_10000.csv",
    "accuracy_10000.json"
]:

    p = OUT / name

    print(
        name,
        p.stat().st_size,
        "bytes"
    )

Arquivos salvos:
predictions_accuracy_10000.npy 10128 bytes
logits_accuracy_10000.npy 400128 bytes
confusion_matrix_10000.npy 928 bytes
confusion_matrix_10000.csv 276 bytes
accuracy_10000.json 627 bytes


In [8]:
print(
    "Pré-empacotando as 10.000 imagens..."
)

packed_all = np.empty(
    (N_IMAGES, 1024, 4),
    dtype=np.uint32
)

t_pack0 = time.perf_counter()


for i in range(N_IMAGES):

    pack_image_to(
        x_test[i],
        packed_all[i]
    )

    if (
        (i + 1) % 1000
        == 0
    ):

        print(
            f"{i+1:5d}/{N_IMAGES}"
        )


pack_elapsed = (
    time.perf_counter()
    - t_pack0
)


print("\nPré-empacotamento concluído.")

print(
    f"Tempo: {pack_elapsed:.2f} s"
)

print(
    "Memória:",
    f"{packed_all.nbytes / 1024**2:.2f} MiB"
)

Pré-empacotando as 10.000 imagens...
 1000/10000
 2000/10000
 3000/10000
 4000/10000
 5000/10000
 6000/10000
 7000/10000
 8000/10000
 9000/10000
10000/10000

Pré-empacotamento concluído.
Tempo: 3.88 s
Memória: 156.25 MiB


In [9]:
TEST_INDICES = [
    0,
    1,
    2,
    123,
    999,
    4321,
    9999
]


print(
    "Validando cache pré-empacotado..."
)


for idx in TEST_INDICES:

    input_buffer[:] = (
        packed_all[idx]
    )

    dma_inference_only()

    pred = int(
        np.argmax(
            decode_logits()
        )
    )

    ref = int(
        predictions_accuracy[idx]
    )

    print(
        f"{idx:5d} | "
        f"pred={pred} | "
        f"ref={ref}"
    )

    assert pred == ref


assert dma_ok()

print("\nPACKED CACHE: OK")

Validando cache pré-empacotado...
    0 | pred=3 | ref=3
    1 | pred=8 | ref=8
    2 | pred=8 | ref=8
  123 | pred=2 | ref=2
  999 | pred=8 | ref=8
 4321 | pred=5 | ref=5
 9999 | pred=7 | ref=7

PACKED CACHE: OK


In [10]:
SEED = 20260825

N_CYCLES = 100

PREFIXES = [
    10,
    20,
    50,
    100
]


# Latência DMA→FPGA→DMA
latencies_ms = np.empty(
    (
        N_CYCLES,
        N_IMAGES
    ),
    dtype=np.float32
)


# Throughput efetivo por ciclo
cycle_wall_s = np.empty(
    N_CYCLES,
    dtype=np.float64
)

cycle_fps_effective = np.empty(
    N_CYCLES,
    dtype=np.float64
)


# Taxa equivalente pela média das
# latências de cada ciclo
cycle_fps_path = np.empty(
    N_CYCLES,
    dtype=np.float64
)


cycle_latency_mean_ms = np.empty(
    N_CYCLES,
    dtype=np.float64
)


cycle_mismatches = np.zeros(
    N_CYCLES,
    dtype=np.int64
)


print(
    "Benchmark configurado:"
)

print(
    N_IMAGES,
    "imagens ×",
    N_CYCLES,
    "ciclos =",
    N_IMAGES*N_CYCLES,
    "inferências"
)

Benchmark configurado:
10000 imagens × 100 ciclos = 1000000 inferências


In [11]:
print("Warm-up do benchmark...")

rng_warmup = np.random.default_rng(
    SEED
)

warmup_order = rng_warmup.permutation(
    N_IMAGES
)[:WARMUP]


for idx in warmup_order:

    input_buffer[:] = (
        packed_all[idx]
    )

    dma_inference_only()

    _ = decode_logits()


assert dma_ok()

print(
    f"Warm-up concluído: "
    f"{WARMUP} inferências."
)

Warm-up do benchmark...
Warm-up concluído: 100 inferências.


In [12]:
print(
    "\n"
    + "=" * 72
)

print(
    "BENCHMARK FINAL ZCU104 — "
    "10.000 IMAGENS × 100 CICLOS"
)

print(
    "=" * 72
)


benchmark_start = (
    time.perf_counter()
)


for cycle in range(N_CYCLES):

    # Ordem determinística diferente
    # para cada ciclo.
    rng_cycle = np.random.default_rng(
        SEED + cycle
    )

    order = rng_cycle.permutation(
        N_IMAGES
    )


    mismatch = 0


    # ========================================================
    # TIMER DO CICLO:
    #
    # inclui:
    #   input_buffer[:] = packed
    #   DMA + FPGA
    #   decode
    #   argmax
    #   loop Python
    #
    # NÃO inclui:
    #   /255
    #   quantização
    #   packing
    # ========================================================

    cycle_t0 = time.perf_counter()


    for j, idx in enumerate(order):

        # -----------------------------------------------
        # Entrada pré-processada.
        # A cópia entra no throughput efetivo,
        # mas fica FORA da latência individual.
        # -----------------------------------------------

        input_buffer[:] = (
            packed_all[idx]
        )


        # -----------------------------------------------
        # LATÊNCIA INFERENCE-ONLY
        #
        # DMA → FPGA → DMA → wait
        # -----------------------------------------------

        t0 = time.perf_counter_ns()

        dma_inference_only()

        t1 = time.perf_counter_ns()


        latencies_ms[
            cycle,
            j
        ] = (
            t1 - t0
        ) / 1_000_000.0


        # -----------------------------------------------
        # Pós-processamento entra no throughput efetivo,
        # mas fica fora da latência acima.
        # -----------------------------------------------

        logits = decode_logits()

        pred = int(
            np.argmax(logits)
        )


        mismatch += int(
            pred
            != int(
                predictions_accuracy[idx]
            )
        )


    cycle_wall = (
        time.perf_counter()
        - cycle_t0
    )


    if not dma_ok():

        raise RuntimeError(
            f"DMA error no ciclo "
            f"{cycle+1}"
        )


    if mismatch != 0:

        raise RuntimeError(
            f"{mismatch} divergências "
            f"no ciclo {cycle+1}"
        )


    mean_lat = float(
        np.mean(
            latencies_ms[cycle]
        )
    )


    fps_effective = (
        N_IMAGES
        / cycle_wall
    )


    fps_path = (
        1000.0
        / mean_lat
    )


    cycle_wall_s[cycle] = (
        cycle_wall
    )

    cycle_fps_effective[cycle] = (
        fps_effective
    )

    cycle_fps_path[cycle] = (
        fps_path
    )

    cycle_latency_mean_ms[cycle] = (
        mean_lat
    )

    cycle_mismatches[cycle] = (
        mismatch
    )


    elapsed_total = (
        time.perf_counter()
        - benchmark_start
    )


    avg_cycle_elapsed = (
        elapsed_total
        / (cycle + 1)
    )


    eta = (
        avg_cycle_elapsed
        * (
            N_CYCLES
            - cycle
            - 1
        )
    )


    print(
        f"{cycle+1:03d}/{N_CYCLES} | "
        f"lat={mean_lat:.6f} ms | "
        f"path={fps_path:.2f} FPS | "
        f"effective={fps_effective:.2f} FPS | "
        f"ETA={eta/60:.1f} min"
    )


benchmark_elapsed = (
    time.perf_counter()
    - benchmark_start
)


print(
    "\nBenchmark concluído."
)

print(
    f"Tempo total: "
    f"{benchmark_elapsed/60:.2f} min"
)


BENCHMARK FINAL ZCU104 — 10.000 IMAGENS × 100 CICLOS
001/100 | lat=0.848675 ms | path=1178.31 FPS | effective=899.80 FPS | ETA=18.3 min
002/100 | lat=0.848449 ms | path=1178.62 FPS | effective=900.63 FPS | ETA=18.1 min
003/100 | lat=0.850021 ms | path=1176.44 FPS | effective=899.36 FPS | ETA=18.0 min
004/100 | lat=0.849270 ms | path=1177.48 FPS | effective=898.92 FPS | ETA=17.8 min
005/100 | lat=0.849866 ms | path=1176.66 FPS | effective=899.70 FPS | ETA=17.6 min
006/100 | lat=0.849579 ms | path=1177.05 FPS | effective=899.41 FPS | ETA=17.4 min
007/100 | lat=0.849322 ms | path=1177.41 FPS | effective=898.51 FPS | ETA=17.2 min
008/100 | lat=0.849326 ms | path=1177.40 FPS | effective=899.03 FPS | ETA=17.1 min
009/100 | lat=0.850012 ms | path=1176.45 FPS | effective=898.45 FPS | ETA=16.9 min
010/100 | lat=0.850078 ms | path=1176.36 FPS | effective=898.49 FPS | ETA=16.7 min
011/100 | lat=0.848997 ms | path=1177.86 FPS | effective=900.34 FPS | ETA=16.5 min
012/100 | lat=0.849119 ms | path=

098/100 | lat=0.846404 ms | path=1181.47 FPS | effective=901.39 FPS | ETA=0.4 min
099/100 | lat=0.848755 ms | path=1178.20 FPS | effective=900.34 FPS | ETA=0.2 min
100/100 | lat=0.848156 ms | path=1179.03 FPS | effective=900.55 FPS | ETA=0.0 min

Benchmark concluído.
Tempo total: 18.54 min


In [13]:
np.save(
    OUT / "latencies_inference_only_10k100.npy",
    latencies_ms
)

np.save(
    OUT / "cycle_wall_s_10k100.npy",
    cycle_wall_s
)

np.save(
    OUT / "fps_effective_10k100.npy",
    cycle_fps_effective
)

np.save(
    OUT / "fps_path_10k100.npy",
    cycle_fps_path
)

np.save(
    OUT / "cycle_latency_mean_ms_10k100.npy",
    cycle_latency_mean_ms
)

np.save(
    OUT / "cycle_mismatches_10k100.npy",
    cycle_mismatches
)


print("Dados brutos salvos.")

Dados brutos salvos.


In [14]:
CSV_PATH = (
    OUT / "cycles_10k100.csv"
)


with open(
    CSV_PATH,
    "w",
    newline=""
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "cycle",
        "images",
        "wall_s",
        "effective_fps",
        "mean_latency_ms",
        "path_equivalent_fps",
        "mismatches"
    ])


    for c in range(N_CYCLES):

        writer.writerow([
            c + 1,
            N_IMAGES,
            cycle_wall_s[c],
            cycle_fps_effective[c],
            cycle_latency_mean_ms[c],
            cycle_fps_path[c],
            cycle_mismatches[c]
        ])


print("CSV salvo:")
print(CSV_PATH)

CSV salvo:
/home/xilinx/jupyter_notebooks/resnet8_hls_ip11/resultados_final_10k100_2026-08-28/cycles_10k100.csv


In [15]:
all_lat = (
    latencies_ms
    .reshape(-1)
    .astype(np.float64)
)


lat_mean = float(
    np.mean(all_lat)
)

lat_median = float(
    np.median(all_lat)
)

lat_std = float(
    np.std(
        all_lat,
        ddof=1
    )
)

lat_cv = (
    100.0
    * lat_std
    / lat_mean
)

lat_p95 = float(
    np.percentile(
        all_lat,
        95
    )
)

lat_p99 = float(
    np.percentile(
        all_lat,
        99
    )
)

lat_min = float(
    np.min(all_lat)
)

lat_max = float(
    np.max(all_lat)
)


latency_stats = {

    "samples":
        int(all_lat.size),

    "mean_ms":
        lat_mean,

    "median_ms":
        lat_median,

    "std_ms":
        lat_std,

    "cv_percent":
        lat_cv,

    "p95_ms":
        lat_p95,

    "p99_ms":
        lat_p99,

    "min_ms":
        lat_min,

    "max_ms":
        lat_max,

    "equivalent_fps_from_mean_latency":
        1000.0 / lat_mean
}


print(
    json.dumps(
        latency_stats,
        indent=2
    )
)

{
  "samples": 1000000,
  "mean_ms": 0.8493000462760926,
  "median_ms": 0.8493599891662598,
  "std_ms": 0.015460288235222407,
  "cv_percent": 1.8203564574158209,
  "p95_ms": 0.862089991569519,
  "p99_ms": 0.8756499886512756,
  "min_ms": 0.8259900212287903,
  "max_ms": 4.762209892272949,
  "equivalent_fps_from_mean_latency": 1177.4401807519948
}


In [16]:
Z95 = 1.959963984540054


def cycle_statistics(x):

    x = np.asarray(
        x,
        dtype=np.float64
    )

    n = len(x)

    mean = float(
        np.mean(x)
    )

    sd = float(
        np.std(
            x,
            ddof=1
        )
    )

    cv = (
        100.0
        * sd
        / mean
    )

    half = (
        Z95
        * sd
        / math.sqrt(n)
    )

    return {
        "n_cycles":
            n,

        "mean":
            mean,

        "sd":
            sd,

        "cv_percent":
            cv,

        "ci95_low":
            mean - half,

        "ci95_high":
            mean + half,

        "min":
            float(np.min(x)),

        "max":
            float(np.max(x))
    }


effective_cycle_stats = (
    cycle_statistics(
        cycle_fps_effective
    )
)

path_cycle_stats = (
    cycle_statistics(
        cycle_fps_path
    )
)


GLOBAL_EFFECTIVE_FPS = (
    N_IMAGES * N_CYCLES
    / np.sum(
        cycle_wall_s
    )
)


print(
    "===== THROUGHPUT ====="
)

print(
    "Média FPS dos ciclos:",
    f"{effective_cycle_stats['mean']:.4f}"
)

print(
    "Global effective FPS:",
    f"{GLOBAL_EFFECTIVE_FPS:.4f}"
)

print(
    "SD:",
    f"{effective_cycle_stats['sd']:.4f}"
)

print(
    "CV:",
    f"{effective_cycle_stats['cv_percent']:.4f}%"
)

print(
    "IC95:",
    f"[{effective_cycle_stats['ci95_low']:.4f}, "
    f"{effective_cycle_stats['ci95_high']:.4f}]"
)

print(
    "\nTaxa equivalente média do path:",
    f"{path_cycle_stats['mean']:.4f} FPS"
)

===== THROUGHPUT =====
Média FPS dos ciclos: 899.2073
Global effective FPS: 899.2063
SD: 0.9882
CV: 0.1099%
IC95: [899.0136, 899.4010]

Taxa equivalente média do path: 1177.4412 FPS


In [17]:
BOOTSTRAP_REPS = 20000

rng_boot = np.random.default_rng(
    SEED + 999999
)


def bootstrap_mean_ci(
    values,
    reps=20000
):

    values = np.asarray(
        values,
        dtype=np.float64
    )

    n = len(values)

    boot_means = np.empty(
        reps,
        dtype=np.float64
    )


    for i in range(reps):

        sample = rng_boot.choice(
            values,
            size=n,
            replace=True
        )

        boot_means[i] = (
            np.mean(sample)
        )


    return (
        float(
            np.percentile(
                boot_means,
                2.5
            )
        ),
        float(
            np.percentile(
                boot_means,
                97.5
            )
        )
    )


boot_low, boot_high = (
    bootstrap_mean_ci(
        cycle_fps_effective,
        BOOTSTRAP_REPS
    )
)


effective_cycle_stats[
    "bootstrap_ci95_low"
] = boot_low

effective_cycle_stats[
    "bootstrap_ci95_high"
] = boot_high


print(
    "Bootstrap IC95:",
    f"[{boot_low:.4f}, "
    f"{boot_high:.4f}] FPS"
)

Bootstrap IC95: [899.0196, 899.4038] FPS


In [18]:
prefix_results = {}


for prefix in PREFIXES:

    fps_cycles = (
        cycle_fps_effective[:prefix]
    )

    wall_cycles = (
        cycle_wall_s[:prefix]
    )

    lat_prefix = (
        latencies_ms[:prefix]
        .reshape(-1)
        .astype(np.float64)
    )


    global_fps = (
        N_IMAGES
        * prefix
        / np.sum(
            wall_cycles
        )
    )


    prefix_results[
        str(prefix)
    ] = {

        "cycles":
            prefix,

        "total_inferences":
            N_IMAGES * prefix,

        "global_effective_fps":
            float(global_fps),

        "mean_cycle_fps":
            float(
                np.mean(
                    fps_cycles
                )
            ),

        "mean_latency_ms":
            float(
                np.mean(
                    lat_prefix
                )
            ),

        "median_latency_ms":
            float(
                np.median(
                    lat_prefix
                )
            ),

        "p95_latency_ms":
            float(
                np.percentile(
                    lat_prefix,
                    95
                )
            ),

        "p99_latency_ms":
            float(
                np.percentile(
                    lat_prefix,
                    99
                )
            )
    }


print(
    "===== CONVERGÊNCIA ====="
)


for prefix in PREFIXES:

    s = prefix_results[
        str(prefix)
    ]

    print(
        f"{prefix:3d} ciclos | "
        f"N={s['total_inferences']:7d} | "
        f"lat={s['mean_latency_ms']:.6f} ms | "
        f"p95={s['p95_latency_ms']:.6f} ms | "
        f"FPS global={s['global_effective_fps']:.3f}"
    )

===== CONVERGÊNCIA =====
 10 ciclos | N= 100000 | lat=0.849460 ms | p95=0.862230 ms | FPS global=899.230
 20 ciclos | N= 200000 | lat=0.849442 ms | p95=0.862120 ms | FPS global=899.136
 50 ciclos | N= 500000 | lat=0.849526 ms | p95=0.862200 ms | FPS global=898.905
100 ciclos | N=1000000 | lat=0.849300 ms | p95=0.862090 ms | FPS global=899.206


In [19]:
FIRST_N = 20
LAST_N = 20


first_mean_fps = float(
    np.mean(
        cycle_fps_effective[
            :FIRST_N
        ]
    )
)

last_mean_fps = float(
    np.mean(
        cycle_fps_effective[
            -LAST_N:
        ]
    )
)

drift_percent = (
    (
        last_mean_fps
        / first_mean_fps
    )
    - 1.0
) * 100.0


effective_cycle_stats[
    "first_20_mean"
] = first_mean_fps

effective_cycle_stats[
    "last_20_mean"
] = last_mean_fps

effective_cycle_stats[
    "drift_percent"
] = drift_percent


print(
    "Primeiros 20:",
    f"{first_mean_fps:.4f} FPS"
)

print(
    "Últimos 20:",
    f"{last_mean_fps:.4f} FPS"
)

print(
    "Drift:",
    f"{drift_percent:+.4f}%"
)

Primeiros 20: 899.1364 FPS
Últimos 20: 900.4819 FPS
Drift: +0.1496%


In [20]:
FINAL_SUMMARY = {

    "experiment":
        "ZCU104 hls4ml final 10k x 100",

    "model":
        "ResNet8 CIFAR-10",

    "model_output":
        "10 logits, no softmax",

    "platform":
        "AMD/Xilinx ZCU104",

    "batch":
        1,

    "clock_mhz":
        100.0,

    "unique_images":
        N_IMAGES,

    "cycles":
        N_CYCLES,

    "total_inferences":
        N_IMAGES * N_CYCLES,

    "warmup":
        WARMUP,

    "seed":
        SEED,

    "latency_boundary":
        (
            "input already copied to PynqBuffer; "
            "timer covers AXI DMA MM2S -> "
            "hls4ml FPGA -> AXI DMA S2MM -> waits"
        ),

    "effective_throughput_boundary":
        (
            "prepacked image copy to PynqBuffer + "
            "DMA/FPGA/DMA + decode logits + argmax "
            "+ Python serial loop"
        ),

    "excluded_from_inference_only":
        (
            "dataset loading, uint8->float32, /255, "
            "fixed-point quantization and packing"
        ),

    "accuracy":
        accuracy_summary,

    "latency_ms":
        latency_stats,

    "effective_throughput_fps":
        effective_cycle_stats,

    "global_effective_fps":
        float(
            GLOBAL_EFFECTIVE_FPS
        ),

    "path_equivalent_fps":
        path_cycle_stats,

    "prefix_results":
        prefix_results,

    "all_cycle_mismatches":
        int(
            np.sum(
                cycle_mismatches
            )
        )
}


(
    OUT / "FINAL_SUMMARY_10k100.json"
).write_text(
    json.dumps(
        FINAL_SUMMARY,
        indent=2
    )
)


print(
    "\n"
    + "=" * 72
)

print(
    "RESULTADO FINAL — "
    "ZCU104 10K × 100"
)

print(
    "=" * 72
)


print(
    f"Accuracy           : "
    f"{accuracy*100:.4f}%"
)

print(
    f"Lat. média         : "
    f"{lat_mean:.6f} ms"
)

print(
    f"Mediana            : "
    f"{lat_median:.6f} ms"
)

print(
    f"p95                : "
    f"{lat_p95:.6f} ms"
)

print(
    f"p99                : "
    f"{lat_p99:.6f} ms"
)

print(
    f"FPS efetivo médio  : "
    f"{effective_cycle_stats['mean']:.4f}"
)

print(
    f"FPS efetivo GLOBAL : "
    f"{GLOBAL_EFFECTIVE_FPS:.4f}"
)

print(
    f"Path equiv. FPS    : "
    f"{path_cycle_stats['mean']:.4f}"
)

print(
    f"CV ciclos          : "
    f"{effective_cycle_stats['cv_percent']:.4f}%"
)

print(
    f"Drift              : "
    f"{drift_percent:+.4f}%"
)

print(
    f"DMA mismatches     : "
    f"{np.sum(cycle_mismatches)}"
)


RESULTADO FINAL — ZCU104 10K × 100
Accuracy           : 74.9200%
Lat. média         : 0.849300 ms
Mediana            : 0.849360 ms
p95                : 0.862090 ms
p99                : 0.875650 ms
FPS efetivo médio  : 899.2073
FPS efetivo GLOBAL : 899.2063
Path equiv. FPS    : 1177.4412
CV ciclos          : 0.1099%
Drift              : +0.1496%
DMA mismatches     : 0


In [21]:
ENVIRONMENT = {

    "real_collection_date":
        "2026-08-28",

    "platform":
        "AMD/Xilinx ZCU104",

    "device":
        "XCZU7EV-FFVC1156-2-E",

    "kernel":
        platform.release(),

    "python":
        platform.python_version(),

    "numpy":
        np.__version__,

    "pynq":
        __import__("pynq").__version__,

    "visible_arm_cpus":
        os.cpu_count(),

    "bitstream":
        BIT.name,

    "hwh":
        HWH.name,

    "dataset":
        DATA.name,

    "dataset_shape":
        list(
            x_test.shape
        ),

    "precision":
        "ap_fixed<22,12,AP_RND_CONV,AP_SAT>",

    "clock_mhz":
        100.0,

    "batch":
        1,

    "unique_images":
        N_IMAGES,

    "cycles":
        N_CYCLES,

    "total_inferences":
        N_IMAGES*N_CYCLES,

    "warmup":
        WARMUP,

    "seed":
        SEED
}


(
    OUT / "ENVIRONMENT.json"
).write_text(
    json.dumps(
        ENVIRONMENT,
        indent=2
    )
)

print("ENVIRONMENT.json salvo.")

ENVIRONMENT.json salvo.


In [22]:
artifacts = [
    BIT,
    HWH,
    DATA
]


for src in artifacts:

    if src.exists():

        dst = (
            OUT
            / src.name
        )

        shutil.copy2(
            src,
            dst
        )

        print(
            "OK:",
            dst.name
        )

    else:

        print(
            "NÃO ENCONTRADO:",
            src
        )

OK: resnet8_hls_ip11.bit
OK: resnet8_hls_ip11.hwh
OK: cifar10_test_uint8.npz


In [23]:
NOTEBOOK = (
    BASE
    / "benchmark_resnet8_hls_ip11.ipynb"
)

if NOTEBOOK.exists():

    shutil.copy2(
        NOTEBOOK,
        OUT / NOTEBOOK.name
    )

    print(
        "Notebook copiado:",
        NOTEBOOK.name
    )

else:

    print(
        "Notebook não encontrado:",
        NOTEBOOK
    )

Notebook copiado: benchmark_resnet8_hls_ip11.ipynb


In [24]:
readme = f"""# ResNet8 hls4ml — Benchmark final ZCU104 10k × 100

## Objetivo

Esta coleta foi realizada para aumentar a paridade do protocolo experimental
entre a ZCU104 e os benchmarks CPU/GPU.

Foram utilizadas as 10.000 imagens oficiais do conjunto de teste CIFAR-10 em
100 ciclos completos, totalizando 1.000.000 de inferências.

## Modelo

- Modelo: ResNet8
- Dataset: CIFAR-10
- Entrada: 32 × 32 × 3
- Classes: 10
- Saída: logits, sem softmax
- Batch: 1
- Plataforma: AMD/Xilinx ZCU104
- Clock do acelerador: 100 MHz
- Precisão: ap_fixed<22,12,AP_RND_CONV,AP_SAT>

## Acurácia

A acurácia foi validada separadamente da coleta de desempenho.

- Imagens únicas: {N_IMAGES}
- Acertos: {correct}
- Acurácia top-1: {accuracy*100:.4f}%
- IC95% Wilson: [{wilson_low*100:.4f}%; {wilson_high*100:.4f}%]

A execução de acurácia não é utilizada como benchmark de latência ou throughput.

## Protocolo de desempenho

- Imagens únicas por ciclo: {N_IMAGES}
- Ciclos: {N_CYCLES}
- Inferências totais: {N_IMAGES*N_CYCLES}
- Warm-up: {WARMUP}
- Seed: {SEED}
- Outliers removidos: nenhum

As imagens foram normalizadas, quantizadas e empacotadas antes da janela
temporizada.

### Latência inference-only

A latência individual mede:

```text
entrada já no PynqBuffer
→ DMA MM2S
→ ResNet8 hls4ml
→ DMA S2MM
→ wait

SyntaxError: incomplete input (136367445.py, line 1)